# 02 — Baseline vs Fine-tuned Comparison
Compare rule-based lexicon matching against the fine-tuned DeBERTa model.

In [ ]:
import json, re
from seqeval.metrics import f1_score, classification_report
from training.data_prep import BIAS_LEXICON, annotate_iob, tokenize_simple

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

test = load_jsonl('data/annotated/test.jsonl')
print(f'Test samples: {len(test)}')

In [ ]:
# Rule-based baseline (lexicon lookup)
true_labels, rule_preds = [], []
for s in test:
    true_labels.append(s['labels'])
    rule_preds.append(annotate_iob(s['tokens'], BIAS_LEXICON))

print('── Rule-based baseline ──')
print(classification_report(true_labels, rule_preds, zero_division=0))
baseline_f1 = f1_score(true_labels, rule_preds, zero_division=0)
print(f'Macro F1: {baseline_f1:.4f}')

In [ ]:
# Fine-tuned model (run training/evaluate.py first)
from transformers import pipeline
import torch

ner = pipeline('token-classification', model='models/deberta-jd-bias-v1',
               aggregation_strategy='simple',
               device=0 if torch.cuda.is_available() else -1)

# Quick check on 3 samples
for s in test[:3]:
    result = ner(s['text'])
    print('TEXT:', s['text'][:100])
    print('PREDS:', [(r['word'], r['entity_group'], round(r['score'],2)) for r in result])
    print()